# Data Understanding

This notebook provides a concise overview of the BRFSS 2015 Diabetes Health Indicators dataset and motivates the clinical risk prediction task used in this project.

The focus is on understanding the target distribution, class imbalance, and the binary risk prediction formulation used for downstream scikit-learn and PyTorch experiments.

In [3]:
import pandas as pd
import numpy as numpy
import matplotlib.pyplot as pyplot
from pathlib import Path

from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/diabetes_012_health_indicators_BRFSS2015.csv")

def load_diabetes_data(data_path=DATA_PATH):
    if not data_path.is_file():
        raise FileNotFoundError(
            f"Dataset not found at {data_path}. "
            "Please place the BRFSS 2015 diabetes CSV file in the data/ directory."
        )
    return pd.read_csv(data_path)

df = load_diabetes_data()

df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [6]:
df.shape

(253680, 22)

In [7]:
df.columns

Index(['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')

In [8]:
df["Diabetes_012"].value_counts().sort_index()

Diabetes_012
0.0    213703
1.0      4631
2.0     35346
Name: count, dtype: int64

In [9]:
df["Diabetes_012"].value_counts(normalize=True).sort_index()

Diabetes_012
0.0    0.842412
1.0    0.018255
2.0    0.139333
Name: proportion, dtype: float64

In [10]:
df["Diabetes_binary"] = (df["Diabetes_012"] > 0).astype(int)

df["Diabetes_binary"].value_counts().sort_index()

Diabetes_binary
0    213703
1     39977
Name: count, dtype: int64

## Target Distribution

The original target variable contains three classes: no diabetes, prediabetes, and diabetes.

Because the prediabetes class is extremely small and the project is framed as a preventative screening task, the target is converted into a binary risk prediction problem:

- 0: No diabetes
- 1: At risk, including prediabetes or diabetes

This binary formulation supports a recall-focused clinical screening objective.